# UFC Fight Prediction (Machine Learning) 

**Goal:** train a model to predict the winner of a UFC fight,
using the features built in the dbt pipeline (ml_fight_dataset).

## 1. Load the data
We read the ML-ready table from DuckDB in read-only mode,
so it doesn't conflict with DBeaver or dbt.

In [ ]:
pip install pandas

In [2]:
import duckdb
import pandas as pd

con = duckdb.connect("data/ufc.duckdb", read_only=True)
df = con.sql("SELECT * FROM ml_fight_dataset").df()
con.close()

print("Shape:", df.shape)
df.columns.tolist()

Shape: (8701, 21)


['fight_id',
 'event_date',
 'fighter_a',
 'a_fights_before',
 'a_wins_before',
 'a_win_rate',
 'a_won_previous',
 'a_days_since_last',
 'a_age',
 'a_reach',
 'a_stance',
 'fighter_b',
 'b_fights_before',
 'b_wins_before',
 'b_win_rate',
 'b_won_previous',
 'b_days_since_last',
 'b_age',
 'b_reach',
 'b_stance',
 'a_won']

## 2. Explore the dataset

Before modeling, we inspect the columns, their types,
and the missing values (NA) we'll need to handle.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8701 entries, 0 to 8700
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   fight_id           8701 non-null   str           
 1   event_date         8701 non-null   datetime64[us]
 2   fighter_a          8701 non-null   str           
 3   a_fights_before    8701 non-null   int64         
 4   a_wins_before      8701 non-null   float64       
 5   a_win_rate         8701 non-null   float64       
 6   a_won_previous     7347 non-null   Int32         
 7   a_days_since_last  7347 non-null   Int64         
 8   a_age              8573 non-null   Int64         
 9   a_reach            7996 non-null   Int32         
 10  a_stance           8634 non-null   str           
 11  fighter_b          8701 non-null   str           
 12  b_fights_before    8701 non-null   int64         
 13  b_wins_before      8701 non-null   float64       
 14  b_win_rate         

## Handle missing values in the new features

- age, reach: fill with the MEDIAN (a typical value, avoids absurd zeros)
- (stance handled separately, as it is text)

In [4]:
# Fill missing numeric profile features with the median (a sensible typical value)
for col in ["a_age", "b_age", "a_reach", "b_reach"]:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

# Also fill the remaining old NA (first fights), as before
df["a_won_previous"]    = df["a_won_previous"].fillna(0)
df["b_won_previous"]    = df["b_won_previous"].fillna(0)
df["a_days_since_last"] = df["a_days_since_last"].fillna(9999)
df["b_days_since_last"] = df["b_days_since_last"].fillna(9999)

# Check remaining NA (only stance should still have some)
df.isnull().sum()

fight_id              0
event_date            0
fighter_a             0
a_fights_before       0
a_wins_before         0
a_win_rate            0
a_won_previous        0
a_days_since_last     0
a_age                 0
a_reach               0
a_stance             67
fighter_b             0
b_fights_before       0
b_wins_before         0
b_win_rate            0
b_won_previous        0
b_days_since_last     0
b_age                 0
b_reach               0
b_stance             96
a_won                 0
dtype: int64

In [5]:
# Fill missing stance with a dedicated "Unknown" category
df["a_stance"] = df["a_stance"].fillna("Unknown")
df["b_stance"] = df["b_stance"].fillna("Unknown")

# One-hot encode the stance columns
df = pd.get_dummies(df, columns=["a_stance", "b_stance"], dtype=int)

# Look at the new columns created
[c for c in df.columns if "stance" in c]

['a_stance_Open Stance',
 'a_stance_Orthodox',
 'a_stance_Sideways',
 'a_stance_Southpaw',
 'a_stance_Switch',
 'a_stance_Unknown',
 'b_stance_Open Stance',
 'b_stance_Orthodox',
 'b_stance_Sideways',
 'b_stance_Southpaw',
 'b_stance_Switch',
 'b_stance_Unknown']

## Difference features on the new numeric features

diff = A - B. Positive = A has the advantage.
Reach and win_rate: higher is better. Age: we'll keep the raw diff
(older can be more experienced OR more declining - the model decides).

In [6]:
# Difference features for the new numeric profile/career features
df["diff_win_rate"] = df["a_win_rate"] - df["b_win_rate"]
df["diff_age"]      = df["a_age"]      - df["b_age"]
df["diff_reach"]    = df["a_reach"]    - df["b_reach"]

# Quick look
df[["diff_win_rate", "diff_age", "diff_reach"]].describe()

,diff_win_rate,diff_age,diff_reach
count,8701.000000,8701.0,8701.0
mean,-0.003548,0.150213,0.044018
std,0.385022,5.281639,3.278319
min,-1.000000,-25.0,-13.0
25%,-0.200000,-3.0,-2.0
50%,0.000000,0.0,0.0
75%,0.200000,4.0,2.0
max,1.000000,23.0,12.0


## Retrain with enriched features

New feature set: career + form + profile (age, reach, stance) + differences.
Same time-based split. We compare accuracy to the previous model (0.576).

In [8]:
# Recreate the original difference features (lost after kernel restart)
df["diff_fights_before"]   = df["a_fights_before"]   - df["b_fights_before"]
df["diff_wins_before"]     = df["a_wins_before"]     - df["b_wins_before"]
df["diff_won_previous"]    = df["a_won_previous"]    - df["b_won_previous"]
df["diff_days_since_last"] = df["a_days_since_last"] - df["b_days_since_last"]

print("Difference features recreated")

Difference features recreated


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Sort chronologically (time-based split)
df_sorted = df.sort_values("event_date").reset_index(drop=True)

# Build the full feature list
stance_cols = [c for c in df_sorted.columns if "stance" in c]

feature_cols = [
    # career / form
    "a_fights_before", "a_wins_before", "a_win_rate", "a_won_previous", "a_days_since_last",
    "b_fights_before", "b_wins_before", "b_win_rate", "b_won_previous", "b_days_since_last",
    # profile
    "a_age", "a_reach", "b_age", "b_reach",
    # differences
    "diff_fights_before", "diff_wins_before", "diff_won_previous", "diff_days_since_last",
    "diff_win_rate", "diff_age", "diff_reach",
] + stance_cols

X = df_sorted[feature_cols]
y = df_sorted["a_won"]

# Time-based split (oldest 80% train, recent 20% test)
split_index = int(len(df_sorted) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# Train
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
baseline = y_test.value_counts(normalize=True).max()

print(f"Number of features: {len(feature_cols)}")
print(f"Model accuracy: {accuracy:.3f}")
print(f"Baseline:       {baseline:.3f}")
print(f"Improvement:    {accuracy - baseline:+.3f}")
print(f"Previous model: 0.576")

Number of features: 33
Model accuracy: 0.590
Baseline:       0.514
Improvement:    +0.076
Previous model: 0.576


C:\projets\ufc-pipeline\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [10]:
from sklearn.preprocessing import StandardScaler

# Standardize features: fit on TRAIN only (avoid leakage), apply to both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Retrain on scaled data
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model accuracy (scaled): {accuracy:.3f}")
print(f"Baseline:                {baseline:.3f}")
print(f"Improvement:             {accuracy - baseline:+.3f}")
print(f"Previous model:          0.576")

Model accuracy (scaled): 0.590
Baseline:                0.514
Improvement:             +0.076
Previous model:          0.576


## Feature importance (on standardized data - now comparable!)

Since features are now standardized (same scale), the weights are
directly comparable. Larger absolute weight = more influential.
Positive = pushes toward "A wins", negative = toward "A loses".

In [11]:
import pandas as pd

# Weights from the model trained on SCALED data
coefficients = pd.DataFrame({
    "feature": feature_cols,
    "weight": model.coef_[0]
})

coefficients["abs_weight"] = coefficients["weight"].abs()
coefficients = coefficients.sort_values("abs_weight", ascending=False)

coefficients[["feature", "weight"]].head(15)

,feature,weight
15,diff_wins_before,0.353499
14,diff_fights_before,-0.278256
26,a_stance_Unknown,-0.243332
6,b_wins_before,-0.225999
5,b_fights_before,0.203937
19,diff_age,-0.167381
32,b_stance_Unknown,0.147884
1,a_wins_before,0.141873
23,a_stance_Sideways,-0.134672
10,a_age,-0.114209


## Try a Random Forest

Tree-based model: handles non-linear relations and correlated features
better than logistic regression. No scaling needed (trees compare thresholds).
We compare to logistic regression (0.590).

In [12]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest: 200 trees, with a depth limit to avoid overfitting
rf_model = RandomForestClassifier(
    n_estimators=200,      # number of trees
    max_depth=10,          # max depth per tree (limits overfitting)
    min_samples_leaf=20,   # min samples per leaf (smooths predictions)
    random_state=42        # reproducibility
)

# Train on RAW data (no scaling needed for trees)
rf_model.fit(X_train, y_train)

# Evaluate
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"Random Forest accuracy: {rf_accuracy:.3f}")
print(f"Baseline:               {baseline:.3f}")
print(f"Logistic Regression:    0.590")
print(f"Improvement vs baseline: {rf_accuracy - baseline:+.3f}")

Random Forest accuracy: 0.593
Baseline:               0.514
Logistic Regression:    0.590
Improvement vs baseline: +0.079


In [13]:
# Check for overfitting: compare train vs test accuracy
rf_train_pred = rf_model.predict(X_train)
rf_train_acc = accuracy_score(y_train, rf_train_pred)

print(f"Random Forest TRAIN accuracy: {rf_train_acc:.3f}")
print(f"Random Forest TEST accuracy:  {rf_accuracy:.3f}")
print(f"Gap (train - test):           {rf_train_acc - rf_accuracy:+.3f}")

Random Forest TRAIN accuracy: 0.715
Random Forest TEST accuracy:  0.593
Gap (train - test):           +0.121


## Reload enriched dataset (with style features)

The dbt pipeline now includes **style features** derived from fight methods:
- `ko_rate`: share of past wins by KO/TKO (striker profile)
- `sub_rate`: share of past wins by submission (grappler profile)

We reload `ml_fight_dataset` from DuckDB to get these new columns,
then rebuild the feature preparation (missing values, encoding, differences)
before retraining.

In [15]:
import duckdb
import pandas as pd

# Reload the enriched ML dataset (read-only mode)
con = duckdb.connect("data/ufc.duckdb", read_only=True)
df = con.sql("SELECT * FROM ml_fight_dataset").df()
con.close()

print("Shape:", df.shape)
[c for c in df.columns if "rate" in c]


Shape: (8701, 25)


['a_win_rate',
 'a_ko_rate',
 'a_sub_rate',
 'b_win_rate',
 'b_ko_rate',
 'b_sub_rate']

## Prepare features (with style features)

Steps:
1. Fill missing values (median for numeric profile features, defaults for first fights)
2. One-hot encode stance
3. Build difference features (A - B), including the new style features

In [17]:
# --- 1. Handle missing values ---
# Numeric profile features: fill with median
for col in ["a_age", "b_age", "a_reach", "b_reach"]:
    df[col] = df[col].fillna(df[col].median())

# First-fight NA
df["a_won_previous"]    = df["a_won_previous"].fillna(0)
df["b_won_previous"]    = df["b_won_previous"].fillna(0)
df["a_days_since_last"] = df["a_days_since_last"].fillna(9999)
df["b_days_since_last"] = df["b_days_since_last"].fillna(9999)

# Stance: fill missing with "Unknown"
df["a_stance"] = df["a_stance"].fillna("Unknown")
df["b_stance"] = df["b_stance"].fillna("Unknown")

# --- 2. One-hot encode stance ---
df = pd.get_dummies(df, columns=["a_stance", "b_stance"], dtype=int)

# --- 3. Difference features (A - B) ---
df["diff_fights_before"]   = df["a_fights_before"]   - df["b_fights_before"]
df["diff_wins_before"]     = df["a_wins_before"]     - df["b_wins_before"]
df["diff_won_previous"]    = df["a_won_previous"]    - df["b_won_previous"]
df["diff_days_since_last"] = df["a_days_since_last"] - df["b_days_since_last"]
df["diff_win_rate"]        = df["a_win_rate"]        - df["b_win_rate"]
df["diff_age"]             = df["a_age"]             - df["b_age"]
df["diff_reach"]           = df["a_reach"]           - df["b_reach"]
# NEW: style difference features
df["diff_ko_rate"]         = df["a_ko_rate"]         - df["b_ko_rate"]
df["diff_sub_rate"]        = df["a_sub_rate"]        - df["b_sub_rate"]

# Check: no missing values left (except possibly in raw stance, now encoded)
print("Missing values total:", df.isnull().sum().sum())

Missing values total: 0


## Retrain with style features

Full feature set now includes style (ko_rate, sub_rate) and their differences.
Same time-based split + standardization. We compare to the previous best (0.590).

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Sort chronologically (time-based split)
df_sorted = df.sort_values("event_date").reset_index(drop=True)

# Full feature list
stance_cols = [c for c in df_sorted.columns if "stance" in c]

feature_cols = [
    # career / form
    "a_fights_before", "a_wins_before", "a_win_rate", "a_won_previous", "a_days_since_last",
    "b_fights_before", "b_wins_before", "b_win_rate", "b_won_previous", "b_days_since_last",
    # style
    "a_ko_rate", "a_sub_rate", "b_ko_rate", "b_sub_rate",
    # profile
    "a_age", "a_reach", "b_age", "b_reach",
    # differences
    "diff_fights_before", "diff_wins_before", "diff_won_previous", "diff_days_since_last",
    "diff_win_rate", "diff_age", "diff_reach",
    "diff_ko_rate", "diff_sub_rate",
] + stance_cols

X = df_sorted[feature_cols]
y = df_sorted["a_won"]

# Time-based split (oldest 80% train, recent 20% test)
split_index = int(len(df_sorted) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# Standardize (fit on train only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
baseline = y_test.value_counts(normalize=True).max()

print(f"Number of features: {len(feature_cols)}")
print(f"Model accuracy: {accuracy:.3f}")
print(f"Baseline:       {baseline:.3f}")
print(f"Improvement:    {accuracy - baseline:+.3f}")
print(f"Previous (no style): 0.590")

Number of features: 39
Model accuracy: 0.593
Baseline:       0.515
Improvement:    +0.079
Previous (no style): 0.590


In [19]:
%pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 2.8 MB/s eta 0:00:36
   ---------------------------------------- 1.0/101.7 MB 2.8 MB/s eta 0:00:36
    --------------------------------------- 1.6/101.7 MB 2.9 MB/s eta 0:00:35
    --------------------------------------- 2.1/101.7 MB 2.9 MB/s eta 0:00:35
   - -------------------------------------- 2.6/101.7 MB 2.7 MB/s eta 0:00:37
   - -------------------------------------- 3.4/101.7 MB 2.7 MB/s eta 0:00:37
   - -------------------------------------- 3.9/101.7 MB 2.8 MB/s eta 0:00:36
   - -------------------------------------- 4.5/101.7 MB 2.8 MB/s eta 0:00:36
   -- ------------------------------------- 5.2/101.7 MB 2.8 MB/s eta 0:00:35
   -- ------------------------------------- 5.5/101.7 MB 2.6 MB/s eta 0:00:37
   -- ------------------------------------- 6.0/101.7 MB 2.6 MB/s eta 0:00:37
   -- ------------------------------------- 6.6/101.7 MB 2.6 MB/s eta 0


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Try XGBoost (gradient boosting)

XGBoost builds trees sequentially, each correcting previous errors.
Often the strongest model on tabular data. No scaling needed (tree-based).
We compare to logistic regression (0.593).

In [20]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# XGBoost classifier (tree-based, no scaling needed -> use raw X)
xgb_model = XGBClassifier(
    n_estimators=300,        # number of boosting rounds (trees)
    max_depth=4,             # shallow trees (avoid overfitting)
    learning_rate=0.05,      # how much each tree corrects (small = careful)
    subsample=0.8,           # use 80% of rows per tree (regularization)
    colsample_bytree=0.8,    # use 80% of features per tree (regularization)
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

# Evaluate
xgb_pred = xgb_model.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)

# Overfitting check
xgb_train_acc = accuracy_score(y_train, xgb_model.predict(X_train))

print(f"XGBoost TEST accuracy:  {xgb_acc:.3f}")
print(f"XGBoost TRAIN accuracy: {xgb_train_acc:.3f}")
print(f"Gap (overfitting?):     {xgb_train_acc - xgb_acc:+.3f}")
print(f"Baseline:               {baseline:.3f}")
print(f"Logistic Regression:    0.593")

XGBoost TEST accuracy:  0.575
XGBoost TRAIN accuracy: 0.754
Gap (overfitting?):     +0.179
Baseline:               0.515
Logistic Regression:    0.593


## Feature selection (cleaning the model)

We inspect the standardized weights of the logistic regression (our best model).
Features with near-zero weights contribute little and are candidates for removal.

Method: identify low-importance features, drop them, retrain, and **measure** the impact.
A cleaner model with fewer features is more robust, faster, and easier to explain.

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import pandas as pd

# Re-train the logistic regression (our best model) on scaled data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# Show ALL weights sorted by absolute importance
coefficients = pd.DataFrame({
    "feature": feature_cols,
    "weight": model.coef_[0]
})
coefficients["abs_weight"] = coefficients["weight"].abs()
coefficients = coefficients.sort_values("abs_weight", ascending=False)

# Display all rows
pd.set_option("display.max_rows", None)
coefficients[["feature", "weight"]]

,feature,weight
19,diff_wins_before,0.347830
18,diff_fights_before,-0.273349
32,a_stance_Unknown,-0.240955
6,b_wins_before,-0.188577
1,a_wins_before,0.173155
23,diff_age,-0.167304
5,b_fights_before,0.166391
38,b_stance_Unknown,0.145384
0,a_fights_before,-0.136761
29,a_stance_Sideways,-0.134499


## Cleaned feature set

We keep the informative features (career, experience, age, win rate, some differences)
and drop near-zero-weight ones (most stances, style differences, redundant reach).
We retrain and measure: a good cleanup keeps accuracy while simplifying the model.

In [24]:
# Cleaned feature set: keep the meaningful ones, drop near-zero-weight noise
feature_cols_clean = [
    # career / form
    "a_fights_before", "a_wins_before", "a_win_rate",
    "b_fights_before", "b_wins_before", "b_win_rate",
    "a_days_since_last", "b_days_since_last",
    # profile (age has signal; reach was weak but keep diff)
    "a_age", "b_age",
    # differences (the strongest signals)
    "diff_fights_before", "diff_wins_before", "diff_win_rate",
    "diff_age", "diff_reach", "diff_days_since_last",
]

X_clean = df_sorted[feature_cols_clean]

# Same time-based split
X_train_c, X_test_c = X_clean.iloc[:split_index], X_clean.iloc[split_index:]

# Standardize
scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled  = scaler_c.transform(X_test_c)

# Train
model_clean = LogisticRegression(max_iter=1000)
model_clean.fit(X_train_c_scaled, y_train)

# Evaluate
y_pred_c = model_clean.predict(X_test_c_scaled)
acc_clean = accuracy_score(y_test, y_pred_c)

print(f"Features: {len(feature_cols_clean)} (was 39)")
print(f"Cleaned model accuracy: {acc_clean:.3f}")
print(f"Full model accuracy:    0.593")
print(f"Baseline:               {baseline:.3f}")

Features: 16 (was 39)
Cleaned model accuracy: 0.596
Full model accuracy:    0.593
Baseline:               0.515


## Prediction function: who wins between X and Y?

To predict a fight between two named fighters, we need each fighter's
**most recent known features** (their latest state in `fct_fighter_features`).

We load one row per fighter (their most recent fight), which serves as a
lookup table of "current" fighter stats.

> **Note:** these are the features *as of each fighter's last fight*, used as a
> proxy for their current state. This is an approximation suitable for a demo.

In [25]:
import duckdb
import pandas as pd

# Load each fighter's MOST RECENT feature row (their current state)
con = duckdb.connect("data/ufc.duckdb", read_only=True)

latest = con.sql("""
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY fighter ORDER BY event_date DESC) AS rn
        FROM fct_fighter_features
    )
    WHERE rn = 1
""").df()

con.close()

print("Fighters available:", latest.shape[0])
latest[["fighter", "wins_before", "win_rate", "age_at_fight", "reach"]].head()

Fighters available: 2695


,fighter,wins_before,win_rate,age_at_fight,reach
0,AJ Fletcher,1.0,0.333333,26,67
1,Akbarh Arreola,1.0,0.333333,32,71
2,Alan Baudot,0.0,0.000000,34,79
3,Alberto Uda,0.0,0.000000,32,<NA>
4,Alex Oliveira,11.0,0.523810,34,76


## The predict_fight function

Given two fighter names, this function:
1. looks up each fighter's latest features
2. builds the 16 model features for the matchup (including differences)
3. applies the same scaler used in training
4. returns the predicted winner and the probability

In [26]:
import numpy as np

def predict_fight(name_a, name_b):
    # 1. Look up each fighter's latest features
    fa = latest[latest["fighter"] == name_a]
    fb = latest[latest["fighter"] == name_b]

    if fa.empty:
        return f"Fighter not found: {name_a}"
    if fb.empty:
        return f"Fighter not found: {name_b}"

    fa = fa.iloc[0]  # take the single row
    fb = fb.iloc[0]

    # 2. Build the 16 features (same as feature_cols_clean, same order)
    row = {
        "a_fights_before": fa["fights_before"], "a_wins_before": fa["wins_before"], "a_win_rate": fa["win_rate"],
        "b_fights_before": fb["fights_before"], "b_wins_before": fb["wins_before"], "b_win_rate": fb["win_rate"],
        "a_days_since_last": fa["days_since_last_fight"], "b_days_since_last": fb["days_since_last_fight"],
        "a_age": fa["age_at_fight"], "b_age": fb["age_at_fight"],
        "diff_fights_before": fa["fights_before"] - fb["fights_before"],
        "diff_wins_before":   fa["wins_before"]   - fb["wins_before"],
        "diff_win_rate":      fa["win_rate"]      - fb["win_rate"],
        "diff_age":           fa["age_at_fight"]  - fb["age_at_fight"],
        "diff_reach":         (fa["reach"] if pd.notna(fa["reach"]) else 71) - (fb["reach"] if pd.notna(fb["reach"]) else 71),
        "diff_days_since_last": fa["days_since_last_fight"] - fb["days_since_last_fight"],
    }

    # 3. Make a DataFrame in the exact column order, fill any NA, scale
    X_row = pd.DataFrame([row])[feature_cols_clean].fillna(0)
    X_row_scaled = scaler_c.transform(X_row)

    # 4. Predict probability that A wins
    proba_a = model_clean.predict_proba(X_row_scaled)[0][1]

    winner = name_a if proba_a >= 0.5 else name_b
    win_proba = proba_a if proba_a >= 0.5 else 1 - proba_a

    return f"{winner} wins (probability: {win_proba:.1%})"


# Test it!
print(predict_fight("Charles Oliveira", "Alex Oliveira"))

Charles Oliveira wins (probability: 65.5%)


In [27]:
print(predict_fight("Ilia Topuria", "Max Holloway"))

Ilia Topuria wins (probability: 51.6%)


In [28]:
print(predict_fight("Jon Jones", "Stipe Miocic"))

Jon Jones wins (probability: 73.8%)


In [29]:
print(predict_fight("Islam Makhachev", "Charles Oliveira"))

Islam Makhachev wins (probability: 60.4%)


In [30]:
print(predict_fight("Conor McGregor", "Dustin Poirier"))

Dustin Poirier wins (probability: 52.9%)


## Explainable prediction

We upgrade the function to show:
1. A side-by-side comparison of both fighters' key stats
2. The predicted winner and probability
3. The main factors that drove the prediction (feature contributions)

This turns the model from a black box into an interpretable decision aid.

In [31]:
def predict_fight_explained(name_a, name_b):
    fa = latest[latest["fighter"] == name_a]
    fb = latest[latest["fighter"] == name_b]
    if fa.empty: return print(f"Fighter not found: {name_a}")
    if fb.empty: return print(f"Fighter not found: {name_b}")
    fa, fb = fa.iloc[0], fb.iloc[0]

    # --- Build the feature row (same 16 features, same order) ---
    reach_a = fa["reach"] if pd.notna(fa["reach"]) else 71
    reach_b = fb["reach"] if pd.notna(fb["reach"]) else 71
    row = {
        "a_fights_before": fa["fights_before"], "a_wins_before": fa["wins_before"], "a_win_rate": fa["win_rate"],
        "b_fights_before": fb["fights_before"], "b_wins_before": fb["wins_before"], "b_win_rate": fb["win_rate"],
        "a_days_since_last": fa["days_since_last_fight"], "b_days_since_last": fb["days_since_last_fight"],
        "a_age": fa["age_at_fight"], "b_age": fb["age_at_fight"],
        "diff_fights_before": fa["fights_before"] - fb["fights_before"],
        "diff_wins_before":   fa["wins_before"]   - fb["wins_before"],
        "diff_win_rate":      fa["win_rate"]      - fb["win_rate"],
        "diff_age":           fa["age_at_fight"]  - fb["age_at_fight"],
        "diff_reach":         reach_a - reach_b,
        "diff_days_since_last": fa["days_since_last_fight"] - fb["days_since_last_fight"],
    }
    X_row = pd.DataFrame([row])[feature_cols_clean].fillna(0)
    X_row_scaled = scaler_c.transform(X_row)

    # --- Prediction ---
    proba_a = model_clean.predict_proba(X_row_scaled)[0][1]
    winner = name_a if proba_a >= 0.5 else name_b
    win_proba = proba_a if proba_a >= 0.5 else 1 - proba_a

    # --- 1. Side-by-side comparison ---
    print("="*50)
    print(f"  {name_a}  VS  {name_b}")
    print("="*50)
    print(f"{'Stat':<20}{name_a[:12]:>14}{name_b[:12]:>14}")
    print("-"*48)
    print(f"{'Wins':<20}{fa['wins_before']:>14.0f}{fb['wins_before']:>14.0f}")
    print(f"{'Total fights':<20}{fa['fights_before']:>14.0f}{fb['fights_before']:>14.0f}")
    print(f"{'Win rate':<20}{fa['win_rate']:>14.1%}{fb['win_rate']:>14.1%}")
    print(f"{'Age':<20}{fa['age_at_fight']:>14.0f}{fb['age_at_fight']:>14.0f}")
    print(f"{'Reach (in)':<20}{reach_a:>14.0f}{reach_b:>14.0f}")

    # --- 2. Prediction ---
    print("="*50)
    print(f"  PREDICTION: {winner} wins ({win_proba:.1%})")
    print("="*50)

    # --- 3. Key factors (feature contribution = weight x scaled value) ---
    contributions = model_clean.coef_[0] * X_row_scaled[0]
    contrib = pd.DataFrame({"feature": feature_cols_clean, "contribution": contributions})
    contrib["abs"] = contrib["contribution"].abs()
    contrib = contrib.sort_values("abs", ascending=False).head(4)
    print("Top factors (positive favors A, negative favors B):")
    for _, r in contrib.iterrows():
        print(f"  {r['feature']:<22} {r['contribution']:+.3f}")


# Test
predict_fight_explained("Charles Oliveira", "Ilia Topuria")

  Charles Oliveira  VS  Ilia Topuria
Stat                  Charles Oliv  Ilia Topuria
------------------------------------------------
Wins                            24             8
Total fights                    36             8
Win rate                     66.7%        100.0%
Age                             37            28
Reach (in)                      74            69
  PREDICTION: Ilia Topuria wins (55.6%)
Top factors (positive favors A, negative favors B):
  diff_wins_before       +1.470
  diff_fights_before     -1.262
  a_wins_before          +0.764
  a_fights_before        -0.541


## Reload with weight-class change feature

The pipeline now includes `weight_change`: whether a fighter moved **up** (+1),
**down** (-1), or **stayed** (0) in weight division compared to their previous fight
(men and women on separate scales; NULL for first fights or unranked categories).

We reload `ml_fight_dataset` to get this feature, then measure its impact.

In [32]:
import duckdb
import pandas as pd

con = duckdb.connect("data/ufc.duckdb", read_only=True)
df = con.sql("SELECT * FROM ml_fight_dataset").df()
con.close()

print("Shape:", df.shape)
[c for c in df.columns if "weight_change" in c]

Shape: (8701, 27)


['a_weight_change', 'b_weight_change']

## Prepare features (with weight_change) and retrain

We rebuild the full preparation including the weight-change feature,
then retrain the cleaned logistic model to measure its impact (was 0.596).

In [33]:
# --- Missing values ---
for col in ["a_age", "b_age", "a_reach", "b_reach"]:
    df[col] = df[col].fillna(df[col].median())

df["a_won_previous"]    = df["a_won_previous"].fillna(0)
df["b_won_previous"]    = df["b_won_previous"].fillna(0)
df["a_days_since_last"] = df["a_days_since_last"].fillna(9999)
df["b_days_since_last"] = df["b_days_since_last"].fillna(9999)

# weight_change: NULL -> 0 (no known change)
df["a_weight_change"] = df["a_weight_change"].fillna(0)
df["b_weight_change"] = df["b_weight_change"].fillna(0)

# --- Difference features ---
df["diff_fights_before"]   = df["a_fights_before"]   - df["b_fights_before"]
df["diff_wins_before"]     = df["a_wins_before"]     - df["b_wins_before"]
df["diff_win_rate"]        = df["a_win_rate"]        - df["b_win_rate"]
df["diff_age"]             = df["a_age"]             - df["b_age"]
df["diff_reach"]           = df["a_reach"]           - df["b_reach"]
df["diff_days_since_last"] = df["a_days_since_last"] - df["b_days_since_last"]
df["diff_weight_change"]   = df["a_weight_change"]   - df["b_weight_change"]

print("Missing values total:", df.isnull().sum().sum())
print("weight_change distribution (A):")
print(df["a_weight_change"].value_counts())

Missing values total: 163
weight_change distribution (A):
a_weight_change
0     8044
-1     392
1      265
Name: count, dtype: Int64


In [34]:
df.isnull().sum()[df.isnull().sum() > 0]

a_stance    67
b_stance    96
dtype: int64

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

df_sorted = df.sort_values("event_date").reset_index(drop=True)

feature_cols_v2 = [
    "a_fights_before", "a_wins_before", "a_win_rate",
    "b_fights_before", "b_wins_before", "b_win_rate",
    "a_days_since_last", "b_days_since_last",
    "a_age", "b_age",
    "diff_fights_before", "diff_wins_before", "diff_win_rate",
    "diff_age", "diff_reach", "diff_days_since_last",
    "diff_weight_change",
]

X = df_sorted[feature_cols_v2]
y = df_sorted["a_won"]

split_index = int(len(df_sorted) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

acc = accuracy_score(y_test, model.predict(X_test_s))
baseline = y_test.value_counts(normalize=True).max()

print(f"Features: {len(feature_cols_v2)}")
print(f"Accuracy: {acc:.3f}")
print(f"Baseline: {baseline:.3f}")
print(f"Without weight_change: 0.596")

Features: 17
Accuracy: 0.596
Baseline: 0.515
Without weight_change: 0.596
